# Pick your Judge — L2 lab

The same AI answer scored by different judges, different rubrics, different re-runs. The judge is an instrument with its own noise — and the score you report depends on which judge you happened to pick.

Source of uncertainty in focus: **variance around a score.**

## Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
load_dotenv()

from lab import load_examples, score_all

df = load_examples()
print(f'{len(df)} examples loaded')
df.head(6)

## Demo A — Different judges, same answers

Run three judges over the same set of question / AI answer / reference triples:

- **Strict** — binary 0/1, exact match only
- **Lenient** — binary 0/1, accepts paraphrases that contain the key information
- **Scale 1–5** — finer-grained ordinal scale

Watch the same `(question, answer)` pair score wildly differently.

In [ ]:
strict_df  = score_all(df, 'strict')
lenient_df = score_all(df, 'lenient')
scale_df   = score_all(df, 'scale_1_5')

summary = df[['question', 'ai_answer']].copy()
summary['strict']    = strict_df['score'].values
summary['lenient']   = lenient_df['score'].values
summary['scale_1_5'] = scale_df['score'].values
summary

In [ ]:
# Aggregate: 'overall accuracy' under each judge
print(f"strict   pass rate: {summary['strict'].mean():.3f}")
print(f"lenient  pass rate: {summary['lenient'].mean():.3f}")
print(f"scale_1_5 mean:     {summary['scale_1_5'].mean():.3f} (out of 5)")

# Disagreement: rows where strict and lenient differ
disagree = summary[summary['strict'] != summary['lenient']]
print(f'\n{len(disagree)} rows where strict and lenient disagree (out of {len(summary)}):')
print(disagree[['question','ai_answer','strict','lenient','scale_1_5']])

**What this shows.** Two well-defined judges, applied to the same answers, give very different headline numbers. "Accuracy" is not a property of the system — it's a joint property of the system and the judge.

The "right" answers don't change. The number does.

## Demo B — Re-run variance for the same judge

Now pick one judge and run it five times per example with `temperature=0.7`. With a non-zero temperature, the LLM judge's output is no longer deterministic — same input, different score on different calls.

*Requires `LIVE=true` in `.env` and an API key. With the mock fallback this demo trivially returns the same number each time — re-run variance only shows up with a real LLM.*

In [ ]:
import os
if os.getenv('LIVE', 'false').lower() != 'true':
    print('LIVE != true — skipping. Set LIVE=true in .env and add an API key to see judge re-run variance.')
else:
    reruns = score_all(df, 'scale_1_5', temperature=0.7, n_reruns=5)
    spread = reruns.groupby('example_id')['score'].agg(['mean','std','min','max']).round(2)
    spread['question']  = df['question'].values
    spread['ai_answer'] = df['ai_answer'].values
    print(spread[['question','ai_answer','mean','std','min','max']])

**What this shows.** Even with the *same judge* and the *same example*, repeated calls give different scores when the LLM is non-deterministic. The variance is the judge's own noise — not a property of the AI under test. If your eval re-runs the judge once per example, that single number is one draw of a re-run distribution.

Two mitigations: (1) call the judge at `temperature=0` so re-runs are (mostly) deterministic; (2) average across multiple judge re-runs to reduce variance — but recognize each one is one sample.

## The L2 takeaway

1. **The score depends on the judge.** Same answers, different rubrics → different headline numbers. The judge is part of the measurement instrument, not an objective oracle.
2. **The score depends on the rubric's writing.** Strict vs lenient is the same idea written differently — and the numbers diverge.
3. **The score depends on which re-run you happened to look at** (with a non-zero temperature). The same judge on the same example wobbles.

All three are the L2 dimension: **variance around a score** — the instrument has its own noise, distinct from the L1 sampling noise that comes from picking different cases.